<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 00. LeNet-5 — Gradient-Based Learning Applied to Document Recognition

- **원 논문:** [Gradient-Based Learning Applied to Document Recognition](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf)
- **저자 / 발표:** Yann LeCun, Léon Bottou, Yoshua Bengio, and Patrick Haffner · Proceedings of the IEEE 86(11) (1998)
- **난이도 / 예상 시간:** 입문 · 약 70분
- **선수 지식:** PyTorch tensor, Conv2d, pooling, cross-entropy의 기초

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

32×32 입력에서 C1-S2-C3-S4-C5-F6으로 이어지는 LeNet-5의 공간 크기 변화를 재현하고, 공유 합성곱 필터와 평균 subsampling을 사용한 작은 분류기를 합성 패턴에 학습한다. 각 층의 shape, 유한한 loss, 한 번 이상의 loss 감소를 검증한다.

**원 논문과 다른 것**

원 논문은 MNIST 및 문서 인식 파이프라인, 부분 연결 C3, 학습 가능한 subsampling 계수, 84차원 F6과 Euclidean RBF 출력까지 사용한다. 여기서는 다운로드 없는 20개 합성 영상, 완전 연결 C3, AvgPool2d, 10-way linear logits로 핵심 계층 구조만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²) | `LeNet5Mini.c1`, `s2`와 shape 추적 셀 | assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인 |
| §II.B, Fig. 2: C3(16@10²) → S4(16@5²) → C5(120@1²) | `c3`, `s4`, `c5` 및 중간 활성 시각화 | 5×5 valid convolution과 2×2 평균 subsampling의 출력 크기 검증 |
| §II.B ‘LeNet-5’: tanh 비선형성과 F6의 84 units | `torch.tanh`와 `f6 = nn.Linear(120, 84)` | 파라미터/출력 shape와 10-class logits 검증 |
| §II.A ‘Convolutional Networks’: local receptive fields와 shared weights | C1 feature-map 시각화와 합성 위치 패턴 학습 | 동일 Conv2d kernel이 전체 영상 위치에 적용되는 결과를 관찰 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
h^{(l)}_{k,i,j}=\sigma\!\left(
b_k^{(l)}+\sum_c\sum_{u,v}W^{(l)}_{k,c,u,v}
h^{(l-1)}_{c,i+u,j+v}\right)
$$

- $h^{(l)}$은 $l$번째 feature map, $W$는 공유되는 convolution kernel입니다.
- 핵심 역할은 국소 receptive field와 weight sharing으로 공간 패턴을 추출하는 것입니다.
- 코드의 convolution/pooling 모듈이 식의 합과 샘플링을 담당합니다.
- shape은 $[B,C,H,W]\rightarrow[B,C',H',W']$로 변하고 classifier에서 $[B,K]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²)
- **노트북 구현:** `LeNet5Mini.c1`, `s2`와 shape 추적 셀
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 MNIST 및 문서 인식 파이프라인, 부분 연결 C3, 학습 가능한 subsampling 계수, 84차원 F6과 Euclidean RBF 출력까지 사용한다. 여기서는 다운로드 없는 20개 합성 영상, 완전 연결 C3, AvgPool2d, 10-way linear logits로 핵심 계층 구조만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문 구조를 읽는 법

Fig. 2에서 `Cx`는 학습되는 합성곱 층, `Sx`는 해상도를 절반으로 줄이는
subsampling 층입니다. 원 논문의 S2/S4는 2×2 평균에 학습 가능한 계수와 bias를
붙였지만, 이 미니 재현에서는 평균 연산 자체에 초점을 맞춰 `AvgPool2d`를 씁니다.
또한 원 C3의 **부분 연결표**는 완전 연결 합성곱으로 단순화합니다. 그러므로 여기서
확인할 것은 최종 MNIST 정확도가 아니라 Fig. 2의 receptive-field/shape 흐름입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,torch.zeros,torch.randn,torch.stack,torch.tensor,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_position_patterns(repeats=2, size=32):
    # 10개 위치를 서로 다른 class로 쓰는 다운로드 없는 toy digits.
    generator = torch.Generator().manual_seed(101)
    images, labels = [], []
    for repeat in range(repeats):
        for label in range(10):
            image = torch.zeros(1, size, size)
            row = 4 + (label // 5) * 14
            col = 2 + (label % 5) * 6
            image[:, row : row + 7, col : col + 5] = 1.0
            # class마다 작은 가로 획을 더해 단순 위치 복사 이상의 패턴을 만든다.
            image[:, row + label % 4 : row + label % 4 + 2, col : col + 8] = 0.55
            noise = 0.04 * torch.randn(image.shape, generator=generator)
            images.append((image + noise).clamp(0, 1))
            labels.append(label)
    return torch.stack(images), torch.tensor(labels)


lenet_x, lenet_y = make_position_patterns()
lenet_x, lenet_y = ACCELERATOR.move(lenet_x, lenet_y)
assert lenet_x.shape == (20, 1, 32, 32)
assert lenet_y.tolist() == list(range(10)) * 2
print(ACCELERATOR.summary())
print("synthetic batch:", tuple(lenet_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²) / §II.B, Fig. 2: C3(16@10²) → S4(16@5²) → C5(120@1²) / §II.B ‘LeNet-5’: tanh 비선형성과 F6의 84 units
- **이 셀의 책임:** `LeNet5Mini.c1`, `s2`와 shape 추적 셀 / `c3`, `s4`, `c5` 및 중간 활성 시각화 / `torch.tanh`와 `f6 = nn.Linear(120, 84)`
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인 / 5×5 valid convolution과 2×2 평균 subsampling의 출력 크기 검증 / 파라미터/출력 shape와 10-class logits 검증. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Conv2d,nn.Linear,torch.tanh,F.avg_pool2d,torch.flatten -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Conv2d(in_channels, out_channels, kernel_size, ...)</code></summary>

#### `nn.Conv2d(in_channels, out_channels, kernel_size, ...)`

- **역할:** 2차원 convolution 층을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_{n,o,h,w}=b_o+\sum_{c,i,j}W_{o,c,i,j}x_{n,c,h\cdot s_h+i-p_h,w\cdot s_w+j-p_w}
$$

- **기호:** $n,o,c$는 batch·출력·입력 channel, $i,j$는 kernel 위치입니다.
- **수식 → 코드:** `kernel_size`, `stride`, `padding`이 수식의 합 범위와 입력 좌표를 정합니다.
- **직관:** 같은 작은 filter를 모든 공간 위치에 공유해 지역 패턴을 찾습니다.
- **shape 확인:** `[N,C_in,H,W] -> [N,C_out,H_out,W_out]`; 공간 크기는 stride·padding에 따라 달라집니다.
- **공식 문서:** [nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>torch.tanh(input)</code></summary>

#### `torch.tanh(input)`

- **역할:** 실수를 -1과 1 사이 값으로 바꿉니다.
- **입력·반환:** 실수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** RNN hidden state나 범위가 제한된 출력을 만듭니다.
- **주의점:** 큰 절댓값에서 포화되어 gradient가 작아질 수 있습니다.

##### 관련 수식과 코드 연결

$$
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

- **기호:** $x$는 실수이고 출력은 -1과 1 사이입니다.
- **수식 → 코드:** API가 activation을 0 중심의 제한된 범위로 압축합니다.
- **직관:** 부호는 유지하면서 큰 값의 영향을 제한합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.tanh](https://docs.pytorch.org/docs/stable/generated/torch.tanh.html)

</details>

<details>
<summary><code>F.avg_pool2d(input, kernel_size, stride=None, padding=0, ...)</code></summary>

#### `F.avg_pool2d(input, kernel_size, stride=None, padding=0, ...)`

- **역할:** 각 공간 window의 평균으로 feature map을 축소합니다.
- **입력·반환:** 4D 텐서와 window 설정을 받고 축소된 feature map을 반환합니다.
- **이 셀에서 쓰는 이유:** 지역 정보를 평균내어 downsampling하거나 global average pooling을 구성합니다.
- **주의점:** padding 값을 평균 분모에 포함할지와 출력 공간 크기를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
H_{out}=\left\lfloor\frac{H_{in}+2p-d(k-1)-1}{s}+1\right\rfloor
$$

- **기호:** $k,s,p,d$는 kernel, stride, padding, dilation입니다.
- **수식 → 코드:** 각 window에서 max 또는 평균을 계산하고 stride만큼 이동합니다.
- **직관:** channel 수는 유지하면서 공간 정보를 지역 대표값으로 압축합니다.
- **shape 확인:** `[N,C,H,W] -> [N,C,H_out,W_out]`입니다.
- **공식 문서:** [F.avg_pool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.avg_pool2d.html)

</details>

<details>
<summary><code>torch.flatten(input, start_dim=0, end_dim=-1)</code></summary>

#### `torch.flatten(input, start_dim=0, end_dim=-1)`

- **역할:** 축을 합칩니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [torch.flatten](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class LeNet5Mini(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 6, kernel_size=5)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        self.c5 = nn.Conv2d(16, 120, kernel_size=5)
        self.f6 = nn.Linear(120, 84)
        self.classifier = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.tanh(self.c1(x))
        x = F.avg_pool2d(x, kernel_size=2, stride=2)
        x = torch.tanh(self.c3(x))
        x = F.avg_pool2d(x, kernel_size=2, stride=2)
        x = torch.tanh(self.c5(x))
        x = torch.flatten(x, 1)
        x = torch.tanh(self.f6(x))
        return self.classifier(x)


torch.manual_seed(0)
lenet = LeNet5Mini().to(DEVICE)

### 구현 단계 3 · 평가·완료 증거

- **원문 위치:** §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²) / §II.B, Fig. 2: C3(16@10²) → S4(16@5²) → C5(120@1²) / §II.B ‘LeNet-5’: tanh 비선형성과 F6의 84 units
- **이 셀의 책임:** `LeNet5Mini.c1`, `s2`와 shape 추적 셀 / `c3`, `s4`, `c5` 및 중간 활성 시각화 / `torch.tanh`와 `f6 = nn.Linear(120, 84)`
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인 / 5×5 valid convolution과 2×2 평균 subsampling의 출력 크기 검증 / 파라미터/출력 shape와 10-class logits 검증. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.no_grad -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def trace_lenet_features(model, batch):
    c1 = torch.tanh(model.c1(batch))
    s2 = F.avg_pool2d(c1, 2, 2)
    c3 = torch.tanh(model.c3(s2))
    s4 = F.avg_pool2d(c3, 2, 2)
    c5 = torch.tanh(model.c5(s4))
    return {
        "c1": c1,
        "s2": s2,
        "c3": c3,
        "s4": s4,
        "c5": c5,
    }


with torch.no_grad():
    feature_trace = trace_lenet_features(lenet, lenet_x[:1])
    logits = lenet(lenet_x[:1])

c1 = feature_trace["c1"]
s2 = feature_trace["s2"]
c3 = feature_trace["c3"]
s4 = feature_trace["s4"]
c5 = feature_trace["c5"]

assert c1.shape == (1, 6, 28, 28)
assert s2.shape == (1, 6, 14, 14)
assert c3.shape == (1, 16, 10, 10)
assert s4.shape == (1, 16, 5, 5)
assert c5.shape == (1, 120, 1, 1)
assert logits.shape == (1, 10)

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for channel, axis in enumerate(axes):
    axis.imshow(c1[0, channel].detach().cpu().numpy(), cmap="coolwarm")
    axis.set_title(f"C1-{channel}")
    axis.axis("off")
fig.suptitle("LeNet-5 C1 shared-filter responses")
fig.tight_layout()
plt.show()
print("shape trace:", [tuple(t.shape) for t in (c1, s2, c3, s4, c5)])

## 4. 원 논문과 이 구현의 경계

C3의 모든 출력 map을 모든 S2 map에 연결한 것은 원 논문의 연결표와 다릅니다.
출력도 원 논문의 RBF distance가 아니라 오늘날 익숙한 linear logits입니다. 반면
32×32 입력, 5×5 valid convolution, 2×2 평균 subsampling, `6→16→120→84`라는
핵심 shape 경로는 그대로이므로 합성곱 계층의 설계 아이디어를 검증할 수 있습니다.

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §II.A ‘Convolutional Networks’: local receptive fields와 shared weights
- **이 셀의 책임:** C1 feature-map 시각화와 합성 위치 패턴 학습
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 동일 Conv2d kernel이 전체 영상 위치에 적용되는 결과를 관찰. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters,Module.train,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,Optimizer.step,Tensor.detach,Module.eval,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
optimizer = torch.optim.Adam(lenet.parameters(), lr=0.01)
lenet_losses = []


def train_lenet_step(model, batch, targets, optimizer):
    model.train()
    optimizer.zero_grad()
    logits = model(batch)
    loss = F.cross_entropy(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


for _ in range(15):
    lenet_losses.append(
        train_lenet_step(
            lenet,
            lenet_x,
            lenet_y,
            optimizer,
        )
    )

lenet.eval()
with torch.no_grad():
    lenet_accuracy = (lenet(lenet_x).argmax(1) == lenet_y).float().mean().item()
assert torch.isfinite(torch.tensor(lenet_losses)).all()
assert min(lenet_losses[1:]) < lenet_losses[0]

plt.figure(figsize=(5, 3))
plt.plot(lenet_losses, marker="o", markersize=3)
plt.xlabel("optimization step")
plt.ylabel("cross-entropy")
plt.title("LeNet-5 mini reproduction")
plt.grid(alpha=0.3)
plt.show()
print(
    f"loss {lenet_losses[0]:.3f} → {lenet_losses[-1]:.3f}; "
    f"train accuracy={lenet_accuracy:.1%}"
)

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 MNIST 및 문서 인식 파이프라인, 부분 연결 C3, 학습 가능한 subsampling 계수, 84차원 F6과 Euclidean RBF 출력까지 사용한다. 여기서는 다운로드 없는 20개 합성 영상, 완전 연결 C3, AvgPool2d, 10-way linear logits로 핵심 계층 구조만 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.